[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/b_22_stable_sigmoid.ipynb)

# 🟡 Medium: Numerically Stable Sigmoid

*Core Ops & Layers*
Implement sigmoid and log-sigmoid so that they survive large inputs — in the
**backward** pass, which is where the naive versions actually break.

$$\sigma(x) = \frac{1}{1+e^{-x}}, \qquad
\log\sigma(x) = -\log\!\left(1+e^{-x}\right)$$

### Signature
```python
def stable_sigmoid(x): ...        # elementwise, same shape and dtype as x
def stable_log_sigmoid(x): ...    # elementwise, same shape and dtype as x
```

Both must work on arrays of any shape. Do not call `jax.nn.sigmoid`,
`jax.nn.log_sigmoid` or `jax.scipy.special.expit` — implementing them is the
exercise.

### The forward pass is a red herring
Write `1 / (1 + jnp.exp(-x))` and evaluate it at `x = -800`. In float32
`exp(800)` is `inf`, and `1 / (1 + inf)` is `0.0` — which is the *right answer*.
The naive sigmoid looks completely fine:

```
x       [-800,  -50,     0,      50,   800]
naive   [   0,  1.9e-22, 0.5,     1,     1]     ✅ matches jax.nn.sigmoid exactly
```

Now differentiate it. The chain rule has to divide by that `inf`:

```
grad naive    [ nan,  0,  0.25,  1.9e-22,  0]      ← NaN at x = -800
```

### The obvious fix makes it worse
The textbook branch — one form for each sign — looks like it solves this:

```python
jnp.where(x >= 0, 1 / (1 + jnp.exp(-x)), jnp.exp(x) / (1 + jnp.exp(x)))
```

`jnp.where` is not a branch. **Both sides are evaluated** and then selected
between, so `exp(-x)` still overflows for very negative `x` and `exp(x)` still
overflows for very positive `x`. The forward pass is saved by the select, but
the gradient is not:

```
grad where    [ nan,  1.9e-22,  0.25,  1.9e-22,  nan]     ← now NaN at BOTH ends
```

This is the general trap: `jnp.where(c, f(x), g(x))` computes `g(x)` even where
`c` is true, so a NaN or inf in the *unused* branch still reaches the backward
pass.

### What actually works
Make the exponent never positive, so nothing can overflow in the first place:

```python
z = jnp.exp(-jnp.abs(x))            # always in (0, 1]
jnp.where(x >= 0, 1 / (1 + z), z / (1 + z))
```

Both branches are now finite everywhere, so selecting between them is safe.

For log-sigmoid the naive form is worse still — it breaks in the **forward**
pass, returning `-inf` where the answer is simply `-800`:

```
x               [-800,  -50,   0,      50]
log(sigmoid(x)) [-inf,  -50,  -0.693,   0]     ❌
correct         [-800,  -50,  -0.693,  -1.9e-22]
```

`jnp.logaddexp` already handles this, and $\log\sigma(x) = -\log(1+e^{-x})$.

### Why it matters
`log_sigmoid` is the core of every pairwise preference loss — DPO is
`-log σ(β(Δ_policy - Δ_ref))`. Those logits are unbounded, so a naive
implementation trains fine until one batch produces a large margin and the loss
becomes `inf` or the gradient becomes `NaN`. The model does not recover.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def stable_sigmoid(x):
    """Elementwise sigmoid, finite in both value and gradient for any input."""
    pass  # Replace this


def stable_log_sigmoid(x):
    """Elementwise log(sigmoid(x)), finite in both value and gradient."""
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

x = jnp.array([-800.0, -50.0, 0.0, 50.0, 800.0])

naive = lambda v: 1.0 / (1.0 + jnp.exp(-v))
branch = lambda v: jnp.where(v >= 0, 1.0 / (1.0 + jnp.exp(-v)),
                             jnp.exp(v) / (1.0 + jnp.exp(v)))

print("x                ", x)
print("naive   value    ", naive(x))
print("yours   value    ", stable_sigmoid(x))
print("-- the forward pass does not distinguish them --\n")

for name, f in (("naive ", naive), ("where ", branch), ("yours ", stable_sigmoid)):
    g = jax.grad(lambda v: jnp.sum(f(v)))(x)
    print(f"grad {name}      {g}")

print("\nlog-sigmoid, where the FORWARD pass already breaks:")
print("  log(sigmoid(x))", jnp.log(naive(x)))
print("  yours          ", stable_log_sigmoid(x))

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("stable_sigmoid")

# hint("stable_sigmoid")      # stuck? nudge without the answer
# solution("stable_sigmoid")  # spoiler: the reference implementation
# status()                    # your dashboard across all problems